# Anomaly Detection Training

Train a custom anomaly detector on your Roboflow dataset (folder structure, classes `pass` / `fail`).

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

**After running the install cell, RESTART the runtime** (Runtime > Restart session), then run all cells from the top.

## 1. Download Dataset from Roboflow

Classification project with two classes: `pass` (normal) and `fail` (anomalous).

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("folder")
project_name = project.name.lower().replace(" ", "_")


## 2. Install Dependencies

**After this cell finishes, RESTART the runtime (Runtime > Restart session) then run all cells from the top again.**

In [ ]:
# ── Current Colab (Python 3.13+): stock torch/numpy, NOTHING downgraded ─────
import importlib

def _env_ok(loud=True):
    try:
        import torch, timm, openvino, nncf, onnx, anomalib, albumentations
        from packaging import version as _v
        assert _v.parse(timm.__version__) >= _v.parse("1.0.3")
        if loud:
            print("Environment OK:")
            print(f"  torch:    {torch.__version__}")
            print(f"  timm:     {timm.__version__}")
            print(f"  anomalib: {anomalib.__version__}")
            print(f"  openvino: {openvino.__version__}")
        return True
    except Exception:
        return False

if not _env_ok(loud=True):
    print("Installing (~2-3 min)...")
    !pip -q install "setuptools>=79,<80" "wheel>=0.45"
    !pip -q install anomalib openvino nncf onnx albumentations
    !pip -q install -U "timm>=1.0.3"
    importlib.invalidate_caches()
    assert _env_ok(loud=False), "install did not verify — read the log above; do NOT continue"
    print("\n>>> Runtime -> Restart session, then run all cells from the top <<<")


## 3. Training Parameters

In [ ]:
import os

# ============================================================
# EDIT THIS SECTION
# ============================================================
EPOCHS     = 300
BATCH_SIZE = 32
IMAGE_SIZE = (256, 256)

# OPTIMIZE = True   -> produces a smaller, faster model (recommended for edge deployment)
#                      Falls back automatically if the optimization is unstable.
# OPTIMIZE = False  -> skip optimization, keep the standard model.
OPTIMIZE = True

# ============================================================
# PREPROCESSING - improves low-contrast inspection images.
# Set to None or {} to disable.
# ============================================================
CLAHE = {"clip_limit": 2.0, "tile_size": (8, 8)}

print(f"Epochs:     {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Optimize:   {OPTIMIZE}")
print(f"CLAHE:      {'enabled' if CLAHE else 'disabled'}")


## 4. Consolidate Pass / Fail

The model trains on `pass` images only. `fail` images are kept separately and used to auto-compute the decision threshold at the end.

This cell merges `pass/` and `fail/` folders from Roboflow's `train/` and `valid/` splits into a single `./data_consolidated/` directory. A Roboflow `test/` split is not needed — only `train` and `valid` are read.

In [ ]:
import shutil
import cv2
import numpy as np
from pathlib import Path

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}


def apply_clahe(image_bgr, clip_limit, tile_size):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tuple(tile_size))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)


consolidated_root = "./data_consolidated"
if os.path.exists(consolidated_root):
    shutil.rmtree(consolidated_root)

out_pass = Path(consolidated_root) / "pass"
out_fail = Path(consolidated_root) / "fail"
out_pass.mkdir(parents=True, exist_ok=True)

n_pass = n_fail = 0
for split in ["train", "valid"]:
    split_dir = Path(dataset.location) / split
    if not split_dir.exists():
        continue
    for cls_dir in split_dir.iterdir():
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name.lower()
        if cls_name not in ("pass", "fail"):
            continue
        target = out_pass if cls_name == "pass" else out_fail
        target.mkdir(parents=True, exist_ok=True)
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_EXT:
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            if CLAHE:
                img = apply_clahe(img, **CLAHE)
            dst = target / f"{split}_{img_path.name}"
            cv2.imwrite(str(dst), img)
            if cls_name == "pass":
                n_pass += 1
            else:
                n_fail += 1

print(f"Consolidated: pass={n_pass}, fail={n_fail}")
assert n_pass > 0, "No 'pass' images found - check that your Roboflow classes are named 'pass' and 'fail'."


## 5. Train

In [ ]:
# Suppress Rich pretty-printing that causes recursion in IPython
import sys
sys.setrecursionlimit(10000)
os.environ["DISABLE_RICH_LOGGING"] = "1"

from anomalib.data import Folder
from anomalib.models import Stfpm
from anomalib.engine import Engine
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
import timm
import torch

BACKBONE = "mobilenetv4_conv_small"
# MobileNetV4 conv_small pyramid levels (strides 1/8, 1/16, 1/32 -- channels 64, 96, 128).
# blocks.4 is a 960-channel classifier-head expansion at the same spatial resolution
# as blocks.3, not a pyramid level -- using it tanks STFPM AUROC on real datasets.
LAYERS   = ["blocks.1", "blocks.2", "blocks.4"]

# Build datamodule from consolidated Roboflow folders
fail_dir = Path(consolidated_root) / "fail"
fail_exists = fail_dir.exists() and any(fail_dir.iterdir())
datamodule = Folder(
    name=project_name,
    root=consolidated_root,
    normal_dir="pass",
    abnormal_dir="fail" if fail_exists else None,
    train_batch_size=BATCH_SIZE,
    eval_batch_size=BATCH_SIZE,
)

datamodule.setup()
print(f"Training samples: {len(datamodule.train_data)}")

# Build STFPM model - fail fast if any requested layer is missing.
_probe = timm.create_model(BACKBONE, pretrained=False)
_present = {n for n, _ in _probe.named_modules()}
_missing = [l for l in LAYERS if l not in _present]
del _probe
if _missing:
    raise RuntimeError(
        f"LAYERS references missing modules {_missing} in {BACKBONE}."
    )

model = Stfpm(
    backbone=BACKBONE,
    layers=LAYERS,
)


# Simple epoch logger - bypasses Rich/ProgressBar which causes recursion in IPython
class SimpleEpochLogger(Callback):
    def __init__(self, every_n=10):
        self.every_n = every_n
        self.start_time = None

    def on_train_start(self, trainer, pl_module):
        import time
        self.start_time = time.time()
        print(f"Training started - total epochs: {trainer.max_epochs}")

    def on_train_epoch_end(self, trainer, pl_module):
        import time
        epoch = trainer.current_epoch + 1
        if epoch % self.every_n == 0 or epoch == trainer.max_epochs or epoch == 1:
            loss = trainer.callback_metrics.get("train_loss")
            loss_str = f"{loss.item():.4f}" if loss is not None else "?"
            elapsed = time.time() - self.start_time
            eta = elapsed / epoch * (trainer.max_epochs - epoch)
            print(f"  Epoch {epoch:3d}/{trainer.max_epochs} - train_loss: {loss_str}  "
                  f"elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")


print(f"\nTraining STFPM ({BACKBONE}) for up to {EPOCHS} epochs...")
print("(logs every 10 epochs)\n")

# STFPM training is stochastic - loss can spike mid-run and never recover,
# in which case the LAST epoch is garbage. Save the weights from the lowest-loss
# epoch and reload them before export/eval.
best_ckpt = ModelCheckpoint(
    dirpath="./work_dir/best",
    filename="best",
    monitor="train_loss",
    mode="min",
    save_top_k=1,
    save_weights_only=True,
)
engine = Engine(
    max_epochs=EPOCHS,
    default_root_dir="./work_dir",
    enable_progress_bar=False,
    enable_model_summary=False,
    callbacks=[SimpleEpochLogger(every_n=10), best_ckpt],
)
engine.fit(model=model, datamodule=datamodule)

# Reload best-loss weights for the test/export steps below
if best_ckpt.best_model_path:
    ckpt = torch.load(best_ckpt.best_model_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded best checkpoint: {best_ckpt.best_model_path} "
          f"(train_loss={best_ckpt.best_model_score.item():.4f})")
print(f"\nTraining done (BACKBONE={BACKBONE}).")

# Evaluate if fail examples available
if hasattr(datamodule, "test_data") and datamodule.test_data is not None:
    results = engine.test(model=model, datamodule=datamodule)
    print(f"Test results: {results}")


## 6. Export & Optimize Model

In [ ]:
import torch.onnx
import openvino as ov
import nncf

if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        # torch >= 2.6 defaults dynamo=True; this export path is written
        # against the legacy tracer.
        kwargs["dynamo"] = False
        try:
            return _orig_export(*args, **kwargs)
        except TypeError:      # older torch: no dynamo kwarg
            kwargs.pop("dynamo", None)
            return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True

model.eval()
model.model.eval()

os.makedirs("./export", exist_ok=True)
onnx_path = "./export/model.onnx"

dummy = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1])
torch.onnx.export(
    model.model, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["anomaly_map"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)")

core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "./export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin)/1024/1024:.1f} MB)")

H, W = IMAGE_SIZE
MEAN_255 = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255
STD_255  = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255


def calib_preprocess(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.resize(img, (W, H))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    return ((rgb - MEAN_255) / STD_255).transpose(2, 0, 1)[None]


calib_dir = Path(consolidated_root) / "pass"

calib_imgs = sorted(p for p in calib_dir.glob("*.*") if p.suffix.lower() in IMG_EXT)
calib_tensors = [t for t in (calib_preprocess(p) for p in calib_imgs) if t is not None]
print(f"Calibration samples: {len(calib_tensors)}")
assert len(calib_tensors) > 0, "No calibration images available"

opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    ov_model_q = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_q,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "./export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate)/1024/1024:.1f} MB)")
    print(f"  Size reduction: {100*(1 - os.path.getsize(opt_bin_candidate)/os.path.getsize(std_bin)):.0f}%")

    compiled_opt = core.compile_model(optimized, "CPU")
    real_probe = calib_tensors[0]
    zeros = np.zeros_like(real_probe)
    o_real_opt = np.asarray(list(compiled_opt([real_probe]).values())[0])
    o_zero_opt = np.asarray(list(compiled_opt([zeros]).values())[0])

    def outputs_vary(o1, o2, rtol=0.01):
        m1, m2 = float(o1.max()), float(o2.max())
        scale = max(abs(m1), abs(m2), 1e-8)
        return abs(m1 - m2) / scale > rtol

    if outputs_vary(o_real_opt, o_zero_opt):
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK - packaging optimized version")
    else:
        print("  Optimization unstable (outputs collapsed) - packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} - packaging standard version only.")


## 7. Save & Download Pickle

In [ ]:
import pickle

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

# Auto-compute decision threshold from pass (and fail if present) scores.
compiled_final = core.compile_model(core.read_model(final_xml_path), "CPU")


def score_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.resize(img, (W, H))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    tensor = ((rgb - MEAN_255) / STD_255).transpose(2, 0, 1)[None]
    out = compiled_final([tensor])
    for v in out.values():
        return float(np.asarray(v).max())
    return None


pass_imgs = [p for p in (Path(consolidated_root) / "pass").glob("*.*") if p.suffix.lower() in IMG_EXT][:50]
fail_dir  = Path(consolidated_root) / "fail"
fail_imgs = [p for p in fail_dir.glob("*.*") if p.suffix.lower() in IMG_EXT][:30] if fail_dir.exists() else []

pass_scores = [s for s in (score_image(p) for p in pass_imgs) if s is not None]
fail_scores = [s for s in (score_image(p) for p in fail_imgs) if s is not None]

pass_scores_np = np.array(pass_scores)
fail_scores_np = np.array(fail_scores) if fail_scores else None

pass_p95 = float(np.percentile(pass_scores_np, 95)) if len(pass_scores) else 0.0
pass_p99 = float(np.percentile(pass_scores_np, 99)) if len(pass_scores) else 0.0
fail_p05 = float(np.percentile(fail_scores_np, 5)) if fail_scores else None

print(f"Pass scores ({len(pass_scores)}): p95={pass_p95:.4f}, p99={pass_p99:.4f}")
if fail_scores:
    print(f"Fail scores ({len(fail_scores)}): p05={fail_p05:.4f}, max={fail_scores_np.max():.4f}")


def youden_threshold(pass_arr, fail_arr):
    candidates = np.unique(np.concatenate([pass_arr, fail_arr]))
    best_t, best_j = float(candidates[0]), -1.0
    for t in candidates:
        tpr = float((fail_arr > t).mean())
        fpr = float((pass_arr > t).mean())
        j = tpr - fpr
        if j > best_j:
            best_j, best_t = j, float(t)
    return best_t, best_j


if fail_scores and fail_p05 > pass_p99:
    threshold = (pass_p99 + fail_p05) / 2
    rule = "midpoint(pass_p99, fail_p05)"
elif fail_scores:
    t_youden, j = youden_threshold(pass_scores_np, fail_scores_np)
    lower = pass_p95
    upper = max(fail_p05 * 0.98, lower)
    threshold = float(np.clip(t_youden, lower, upper))
    rule = f"Youden's J (J={j:.3f})"
else:
    threshold = pass_p99 * 1.10
    rule = "pass_p99 * 1.10"

print(f"Threshold: {threshold:.4f}  ({rule})")

# Colors from Roboflow (dict {class_name: hex}). Fallback to a simple palette.
try:
    colors = project.colors
    if not colors:
        raise AttributeError
except Exception:
    colors = {"pass": "#00aa00", "fail": "#cc0000"}

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": ["pass", "fail"],
    "colors": colors,
    "meta": {
        "model": BACKBONE,
        "type": "anom",
        "image_size": list(IMAGE_SIZE),
        "precision": precision_tag,
        "clahe": CLAHE,
        "threshold": float(threshold),
        "threshold_rule": rule,
    },
}

pickle_path = f"./{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"\nPickle contains {variant} model ({len(bin_data)/1024/1024:.1f} MB)")
print(f"Saved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path)/1024/1024:.1f} MB")

try:
    from google.colab import files
    files.download(pickle_path)
except ImportError:
    pass
